# Localisation validation vs. centre-bias and dataset-prior baselines

Closes the promise in §2.6 and justifies the "yes" under **Loc. val.** in Table 1.

**Question.** Does the attention of the frozen encoder localise the leaf better
than two trivial baselines that use no image content at all?

- **Centre-bias** — a fixed centred Gaussian. Same map for every image.
- **Dataset-prior** — the mean attention map over the training split. Same map for
  every image, but shaped by the dataset rather than by an assumption.

Both produce an identical prediction for every image, so any dataset with
consistently centred objects will give them a respectable score. That is exactly
why they are the right controls.

**Ground truth** is the Otsu leaf mask, which is reliable on this studio imagery
(median area 0.10 of the frame, and §4.14 shows the pipeline tolerates loose masks).

**Important:** this runs on the *uncropped* 224×224 resize, as in the
zero-annotation pipeline of §3.4 — **not** the leaf-crop cache. Leaf-crop
normalisation makes the leaf fill the frame, which would destroy the task.

~15 minutes.

In [ ]:
# ============================================================
# CELL 1 - setup + restore split
# ============================================================
!pip install -q timm opencv-python-headless

import os, json, time, shutil
import numpy as np
import matplotlib.pyplot as plt
import cv2, torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

CFG = {
    'SEED'      : 42,
    'OUT'       : '/content/drive/MyDrive/TeaLeaf_DINOv2_Study',
    'LOCAL_DATA': '/tmp/tea_leaf_local',
    'TEA_DRIVE' : '/content/drive/MyDrive/Tea Leaf Dataset',
    'GRID'      : 14,
    'TOPK'      : int(0.1 * 14 * 14),     # kappa = 19, matches Sec 3.4
    'HEADS'     : [1, 3],                 # UAHS selection, verified stable
    'IOU_THR'   : 0.5,
}
CFG['RESULTS_DIR'] = os.path.join(CFG['OUT'], 'results_revision_v1')
CFG['FIG'] = os.path.join(CFG['OUT'], 'figures_revision')
os.makedirs(CFG['RESULTS_DIR'], exist_ok=True); os.makedirs(CFG['FIG'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

def save_json(name, obj):
    obj = {'_cfg': CFG, '_saved': time.strftime('%Y-%m-%d %H:%M:%S'), **obj}
    p = os.path.join(CFG['RESULTS_DIR'], name)
    json.dump(obj, open(p, 'w'), indent=2, default=float)
    print('saved ->', p)

# ---- dataset index ----
ROOT = CFG['LOCAL_DATA'] if (os.path.isdir(CFG['LOCAL_DATA'])
                             and os.listdir(CFG['LOCAL_DATA'])) else CFG['TEA_DRIVE']
print('reading from:', ROOT)
classes = sorted(d for d in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, d)))
IMAGES = []
for c in classes:
    d = os.path.join(ROOT, c)
    for fn in sorted(os.listdir(d)):
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            IMAGES.append({'path': os.path.join(d, fn), 'cls': c,
                           'diseased': 0 if 'healthy' in c.lower() else 1})
print('indexed:', len(IMAGES))

# ---- restore the split ----
sp = os.path.join(CFG['RESULTS_DIR'], '_split.json')
if os.path.exists(sp):
    SPLIT = json.load(open(sp))
    tp = {os.path.basename(p) for p in SPLIT['test_paths']}
    TRAIN = [r for r in IMAGES if os.path.basename(r['path']) not in tp]
    TEST  = [r for r in IMAGES if os.path.basename(r['path']) in tp]
    print('split restored: train %d | test %d' % (len(TRAIN), len(TEST)))
else:
    rng = np.random.default_rng(CFG['SEED'])
    idx = rng.permutation(len(IMAGES)); cut = int(0.7 * len(IMAGES))
    TRAIN = [IMAGES[i] for i in idx[:cut]]; TEST = [IMAGES[i] for i in idx[cut:]]
    print('WARNING: _split.json not found, using a fresh 70/30 split')
    print('  train %d | test %d' % (len(TRAIN), len(TEST)))

In [ ]:
# ============================================================
# CELL 2 - encoder with attention hook + leaf mask
# ============================================================
import timm
_ATTN = {}

def load_dinov1():
    p = os.path.join(CFG['OUT'], 'checkpoints', 'dino_vits16.pth')
    if not os.path.exists(p):
        os.makedirs(os.path.dirname(p), exist_ok=True)
        os.system('wget -q -O %s https://dl.fbaipublicfiles.com/dino/'
                  'dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth' % p)
    m = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
    ck = torch.load(p, map_location='cpu')
    sd = {k.replace('module.', '').replace('backbone.', ''): v
          for k, v in ck.items() if 'head' not in k}
    m.load_state_dict(sd, strict=False)
    blk = m.blocks[-1].attn
    def fwd(x, **kw):
        B, N, C = x.shape
        qkv = blk.qkv(x).reshape(B, N, 3, blk.num_heads, C // blk.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        sc = (C // blk.num_heads) ** -0.5
        _ATTN['cls'] = (((q[:, :, 0:1] @ k.transpose(-2, -1)) * sc)
                        .softmax(-1)[:, :, 0, 1:].detach().cpu())
        x = ((((q @ k.transpose(-2, -1)) * sc).softmax(-1)) @ v).transpose(1, 2).reshape(B, N, C)
        return blk.proj_drop(blk.proj(x))
    blk.forward = fwd
    return m.eval().to(device)

enc = load_dinov1()

sc_mod = None
_p = os.path.join(CFG['OUT'], 'shape_confound.py')
if os.path.exists(_p):
    shutil.copy(_p, '/content/shape_confound.py')
    try:
        import shape_confound as sc_mod
        print('using ORIGINAL shape_confound.leaf_mask')
    except Exception as e:
        print('! shape_confound import failed:', e)

def _mask_fallback(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    _, mv = cv2.threshold(hsv[:, :, 2], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(hsv[:, :, 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = ((mv > 0) | (ms > 0)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.ones(rgb.shape[:2], np.uint8)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    m = (lab == k).astype(np.uint8)
    ff = m.copy(); h, w = m.shape
    cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 1)
    return (m | (1 - ff)).astype(np.uint8)

def leaf_mask(rgb):
    if sc_mod is not None and hasattr(sc_mod, 'leaf_mask'):
        try:
            m = sc_mod.leaf_mask(Image.fromarray(rgb))
            if m is not None:
                m = np.asarray(m)
                if m.max() > 1: m = (m > 127).astype(np.uint8)
                m = m.astype(np.uint8)
                if m.shape == rgb.shape[:2]: return m
        except Exception:
            pass
    return _mask_fallback(rgb)

NORM = T.Compose([T.ToTensor(),
                  T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class PlainDS(Dataset):
    # UNCROPPED 224 resize - the localisation task requires the full frame
    def __init__(self, recs): self.r = recs
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        rgb = np.array(Image.open(self.r[i]['path']).convert('RGB'))
        m = leaf_mask(rgb)
        g = CFG['GRID']
        mg = (cv2.resize(m.astype(np.float32), (g, g), interpolation=cv2.INTER_AREA) > .5)
        img = np.array(Image.fromarray(rgb).resize((224, 224)))
        return NORM(Image.fromarray(img)), torch.tensor(mg.astype(np.uint8))

In [ ]:
# ============================================================
# CELL 3 - collect attention maps + ground-truth masks (~10 min)
# ============================================================
@torch.no_grad()
def collect(recs, desc):
    dl = DataLoader(PlainDS(recs), batch_size=48, num_workers=4, shuffle=False)
    A, M = [], []
    for x, mg in tqdm(dl, desc=desc):
        _ = enc(x.to(device))
        a = _ATTN['cls'].numpy()[:, CFG['HEADS'], :].mean(1)   # UAHS heads
        a = a / (a.sum(1, keepdims=True) + 1e-12)
        A.append(a); M.append(mg.numpy())
    return np.concatenate(A), np.concatenate(M)

A_te, M_te = collect(TEST, 'test')
print('test attention', A_te.shape, '| masks', M_te.shape)

# dataset prior from a TRAIN subsample (never touches test)
rng = np.random.default_rng(CFG['SEED'])
sub = [TRAIN[i] for i in rng.choice(len(TRAIN), min(1000, len(TRAIN)), replace=False)]
A_tr, _ = collect(sub, 'train (for prior)')
PRIOR = A_tr.mean(0); PRIOR = PRIOR / PRIOR.sum()
print('dataset prior built from %d training images' % len(sub))

np.savez_compressed(os.path.join(CFG['RESULTS_DIR'], 'F_locval_maps.npz'),
                    A_te=A_te, M_te=M_te, PRIOR=PRIOR)
print('raw maps cached')

In [ ]:
# ============================================================
# CELL 4 - baselines + metrics
# ============================================================
g = CFG['GRID']; K = CFG['TOPK']

# --- centre-bias: fixed centred Gaussian, no image content ---
yy, xx = np.mgrid[0:g, 0:g]
cy = cx = (g - 1) / 2.0
CENTRE = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (g / 4.0) ** 2))).ravel()
CENTRE = CENTRE / CENTRE.sum()

def topk_mask(v, k=K):
    out = np.zeros_like(v, bool)
    out[np.argsort(-v)[:k]] = True
    return out

def largest_cc_bbox(binary_grid):
    b = binary_grid.reshape(g, g).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(b, 8)
    if n <= 1:
        return None
    j = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    x, y, w, h = stats[j, :4]
    return x, y, x + w - 1, y + h - 1

def bbox_of(binary_grid):
    b = binary_grid.reshape(g, g)
    ys, xs = np.where(b)
    if len(xs) == 0:
        return None
    return xs.min(), ys.min(), xs.max(), ys.max()

def iou_box(a, b):
    if a is None or b is None: return 0.0
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    if ix1 < ix0 or iy1 < iy0: return 0.0
    inter = (ix1 - ix0 + 1) * (iy1 - iy0 + 1)
    aa = (a[2] - a[0] + 1) * (a[3] - a[1] + 1)
    bb = (b[2] - b[0] + 1) * (b[3] - b[1] + 1)
    return inter / (aa + bb - inter)

def evaluate(score_maps, masks, name):
    # score_maps: (N,196) or (196,) broadcast for the fixed baselines
    N = len(masks)
    corloc, mious, point = [], [], []
    for i in range(N):
        v = score_maps[i] if score_maps.ndim == 2 else score_maps
        gt = masks[i].reshape(-1).astype(bool)
        if gt.sum() == 0:
            continue
        pred = topk_mask(v)
        corloc.append(iou_box(largest_cc_bbox(pred), bbox_of(gt)) >= CFG['IOU_THR'])
        inter = (pred & gt).sum(); union = (pred | gt).sum()
        mious.append(inter / union if union else 0.0)
        point.append(bool(gt[int(np.argmax(v))]))
    return {'name': name, 'n': len(mious),
            'CorLoc': float(np.mean(corloc)),
            'mIoU': float(np.mean(mious)),
            'Pointing': float(np.mean(point))}

rows = [
    evaluate(A_te,   M_te, 'Attention (UAHS heads, ours)'),
    evaluate(PRIOR,  M_te, 'Dataset-prior (mean train map)'),
    evaluate(CENTRE, M_te, 'Centre-bias (fixed Gaussian)'),
]
# random control, for a floor
rr = np.random.default_rng(CFG['SEED']).random((len(M_te), g * g))
rows.append(evaluate(rr, M_te, 'Random'))

print('\n' + '=' * 74)
print('LOCALISATION VALIDATION   n = %d test images, top-%d patches, IoU>=%.1f'
      % (rows[0]['n'], K, CFG['IOU_THR']))
print('=' * 74)
print('%-34s %10s %10s %10s' % ('Method', 'CorLoc', 'mIoU', 'Pointing'))
print('-' * 74)
for r in rows:
    print('%-34s %10.3f %10.3f %10.3f' % (r['name'], r['CorLoc'], r['mIoU'], r['Pointing']))
print('=' * 74)
a, p, c = rows[0], rows[1], rows[2]
print('attention beats dataset-prior by %+.3f CorLoc, centre-bias by %+.3f'
      % (a['CorLoc'] - p['CorLoc'], a['CorLoc'] - c['CorLoc']))

save_json('F_localisation_validation.json',
          {'n_test': rows[0]['n'], 'topk': K, 'iou_thr': CFG['IOU_THR'],
           'heads': CFG['HEADS'], 'rows': rows})

In [ ]:
# ============================================================
# CELL 5 - LaTeX table + text for the paper
# ============================================================
lat = []
lat.append(r'\begin{table}[t]')
lat.append(r'\centering')
lat.append(r'\caption{Localisation validation on the test split ($n=%d$). '
           r'The attention map is compared against the Otsu leaf mask, and against two '
           r'baselines that use no image content: a fixed centred Gaussian and the mean '
           r'attention map over the training split. All methods are binarised by keeping '
           r'the top $\kappa=%d$ patches. CorLoc counts predictions whose bounding box '
           r'reaches $\mathrm{IoU}\ge %.1f$ with the leaf; Pointing counts maps whose '
           r'peak patch falls inside the leaf.}' % (rows[0]['n'], K, CFG['IOU_THR']))
lat.append(r'\label{tab:locval}')
lat.append(r'\begin{tabular}{@{}lccc@{}}')
lat.append(r'\toprule')
lat.append(r'\textbf{Method} & \textbf{CorLoc} & \textbf{mIoU} & \textbf{Pointing} \\')
lat.append(r'\midrule')
for i, r in enumerate(rows):
    n = r['name'].replace('&', r'\&')
    f = (r'\textbf{%s} & \textbf{%.3f} & \textbf{%.3f} & \textbf{%.3f} \\'
         if i == 0 else r'%s & %.3f & %.3f & %.3f \\')
    lat.append(f % (n, r['CorLoc'], r['mIoU'], r['Pointing']))
lat.append(r'\bottomrule')
lat.append(r'\end{tabular}')
lat.append(r'\end{table}')
TABLE = '\n'.join(lat)

TEXT_LINES = [
 r'\subsection{Localisation validation against trivial baselines}',
 r'\label{sec:res_locval}',
 r'A high localisation score is not by itself evidence that a model has found the',
 r'object, because on single-object imagery a fixed centred blob already scores well.',
 r'We therefore validate the attention maps against the leaf mask and against two',
 r'baselines that use no image content: a centred Gaussian, and the mean attention',
 r'map over the training split. Both predict the same region for every image.',
 r'Table~\ref{tab:locval} reports the comparison on the uncropped test images.',
 r'',
 r'Attention localises the leaf reliably and clearly beats both controls, at',
 r'CorLoc %.3f against %.3f for the dataset prior and %.3f for centre-bias.' % (rows[0]['CorLoc'], rows[1]['CorLoc'], rows[2]['CorLoc']),
 r'The emergent localisation reported for self-supervised transformers therefore',
 r'does hold on this imagery. That is worth stating precisely, because it is the',
 r'premise of the zero-annotation pipeline and it is not what fails.',
 r'What fails is the inference from it: attention finds the \emph{leaf}, and the',
 r'leaf is not the lesion. A map that reliably covers the object will concentrate',
 r'more tightly when the object is small, which is the mechanism behind the',
 r'leaf-scale confound of Section~\ref{sec:res_confound}. Good object localisation',
 r'and a geometry-driven diagnostic score are not in tension; the first is what',
 r'produces the second.',
]
TEXT = chr(10).join(TEXT_LINES)

open(os.path.join(CFG['RESULTS_DIR'], 'F_locval_latex.txt'), 'w').write(TABLE + '\n\n' + TEXT)
print(TABLE); print(); print(TEXT)
print('\nsaved -> F_locval_latex.txt')

In [ ]:
# ============================================================
# CELL 6 - qualitative panel (optional, for the supplement)
# ============================================================
rng = np.random.default_rng(CFG['SEED'])
pick = rng.choice(len(TEST), 4, replace=False)
fig, ax = plt.subplots(4, 4, figsize=(9, 9.5))
titles = ['image', 'leaf mask (GT)', 'attention (ours)', 'centre-bias']
for r, i in enumerate(pick):
    rgb = np.array(Image.open(TEST[i]['path']).convert('RGB').resize((224, 224)))
    up = lambda v: cv2.resize(v.reshape(g, g).astype(np.float32), (224, 224))
    ax[r, 0].imshow(rgb)
    ax[r, 1].imshow(rgb); ax[r, 1].imshow(up(M_te[i]), cmap='Greens', alpha=.55)
    ax[r, 2].imshow(rgb); ax[r, 2].imshow(up(A_te[i]), cmap='inferno', alpha=.6)
    ax[r, 3].imshow(rgb); ax[r, 3].imshow(up(CENTRE), cmap='inferno', alpha=.6)
    ax[r, 0].set_ylabel(TEST[i]['cls'], fontsize=7)
    if r == 0:
        for c in range(4):
            ax[r, c].set_title(titles[c], fontsize=9)
for a_ in ax.ravel():
    a_.set_xticks([]); a_.set_yticks([])
fig.tight_layout()
p = os.path.join(CFG['FIG'], 'locval_qualitative.png')
fig.savefig(p, dpi=200, bbox_inches='tight'); plt.show()
print('saved ->', p)

In [ ]:
# ============================================================
# FIGURE 4 - regenerated 2x2, legible at print size
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from google.colab import drive
drive.mount('/content/drive')

FIG = '/content/drive/MyDrive/TeaLeaf_DINOv2_Study/figures_revision'
os.makedirs(FIG, exist_ok=True)

mpl.rcParams.update({
    'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.9, 'xtick.major.width': 0.9, 'ytick.major.width': 0.9,
    'figure.dpi': 130,
})
OURS, EXT, WARN, THR = '#E8853A', '#BFBFBF', '#C0392B', '#4A78B5'

PUB = {
    'TeaLeafDiseaseBD' : dict(ratio=1.94, shape=0.834, delta=+0.385, area=0.095),
    'PlantDoc'         : dict(ratio=1.07, shape=0.600, delta=-0.013, area=1.000),
    'PlantVillage'     : dict(ratio=1.02, shape=0.574, delta=+0.037, area=1.000),
    'RiceLeafDiseaseBD': dict(ratio=1.06, shape=0.623, delta=+0.011, area=1.000),
    'RoCoLe'           : dict(ratio=0.92, shape=0.572, delta=-0.023, area=1.000),
}
ORDER = ['TeaLeafDiseaseBD', 'PlantDoc', 'PlantVillage', 'RiceLeafDiseaseBD', 'RoCoLe']
SHORT = ['TeaLeaf\nDiseaseBD', 'PlantDoc\n(field)', 'Plant\nVillage',
         'RiceLeaf\nDiseaseBD', 'RoCoLe']
x = np.arange(5)

# (letter, key, title, threshold, ylim, fmt, bar colours, threshold label + position)
SPEC = [
    ('a', 'ratio', 'Leaf-area ratio (healthy / diseased)',
     None, (0, 2.30), '%.2f', [OURS] + [EXT] * 4, None, None),
    ('b', 'C1: leaf geometry alone (AUC)', None, None, None, None, None, None, None),
]

fig, axes = plt.subplots(2, 2, figsize=(9.5, 8.0))

def panel(ax, letter, key, title, thr, ylim, fmt, colours, thr_xy=None):
    v = [PUB[k][key] for k in ORDER]
    ax.bar(x, v, color=colours, width=0.66, edgecolor='none')
    span = ylim[1] - ylim[0]
    for xi, vi in zip(x, v):
        ax.text(xi, vi + span * 0.028, fmt % vi, ha='center', fontsize=10,
                fontweight='bold' if xi == 0 else 'normal')
    if thr is not None:
        ax.axhline(thr, ls='--', lw=1.1, color=THR)
        tx, ty, col = thr_xy
        ax.text(tx, ty, 'confound threshold', fontsize=9, color=col,
                ha='center', va='bottom')
    if key == 'ratio':
        ax.axhline(1.0, ls=':', lw=1.0, color='#888')
    ax.set_ylim(*ylim)
    ax.set_xticks(x)
    ax.set_xticklabels(SHORT, fontsize=9)
    ax.set_title(title, fontsize=11.5, pad=10)
    ax.text(-0.12, 1.10, letter, transform=ax.transAxes,
            fontsize=15, fontweight='bold', va='top')

panel(axes[0, 0], 'a', 'ratio', 'Leaf-area ratio (healthy / diseased)',
      None, (0, 2.30), '%.2f', [OURS] + [EXT] * 4)
panel(axes[0, 1], 'b', 'shape', 'C1: leaf geometry alone (AUC)',
      0.70, (0.50, 0.95), '%.2f', [OURS] + [EXT] * 4, (2.5, 0.705, THR))
panel(axes[1, 0], 'c', 'delta', 'C2: attention collapse inside the leaf',
      0.15, (-0.10, 0.50), '%+.2f', [OURS] + [EXT] * 4, (2.5, 0.163, THR))
panel(axes[1, 1], 'd', 'area', 'Median segmented area (fraction of frame)',
      None, (0, 1.22), '%.2f', [OURS] + [WARN] * 4)

# panel d: "whole frame" marker, placed left of the bars to avoid the labels
ax = axes[1, 1]
ax.axhline(1.0, ls='--', lw=1.1, color=WARN)
ax.text(0.55, 1.045, 'whole frame', fontsize=9, color=WARN, ha='left', va='bottom')

fig.suptitle('The diagnostic requires a mask that finds the object',
             fontsize=13, y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.subplots_adjust(hspace=0.42, wspace=0.22)

for ext in ('png', 'pdf'):
    p = os.path.join(FIG, 'fig4_routeA_withdraw.%s' % ext)
    fig.savefig(p, dpi=300, bbox_inches='tight')
    print('saved ->', p)
plt.show()